# 02 - Feature Engineering

In [1]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [4]:
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [5]:
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

/var/folders/xp/wdfsp8kx50v61w_6k88wky2c0000gn/T/ipykernel_6950/1479199042.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [6]:
df.drop("customerID", axis=1, inplace=True)

In [7]:
y = df["Churn"]
X = df.drop("Churn", axis=1)

## Label Encoding

In [8]:
binary_cols = [col for col in X.columns if X[col].nunique() == 2 and X[col].dtype == "object"]
binary_cols

['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

In [9]:
le = LabelEncoder()
for col in binary_cols:
    X[col] = le.fit_transform(X[col])

## One-Hot Encoding

In [10]:
cat_cols = [col for col in X.select_dtypes(include="object").columns if col not in binary_cols]
cat_cols

['MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaymentMethod']

In [11]:
X = pd.get_dummies(X, columns=cat_cols, drop_first=False)

In [12]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

In [13]:
df_cleaned = pd.concat([X, y], axis=1)
df_cleaned.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn
0,0,0,1,0,-1.277445,0,1,-1.160323,-0.994242,False,...,False,False,True,False,False,False,False,True,False,No
1,1,0,0,0,0.066327,1,0,-0.259629,-0.173244,True,...,False,False,False,True,False,False,False,False,True,No
2,1,0,0,0,-1.236724,1,1,-0.362660,-0.959674,True,...,False,False,True,False,False,False,False,False,True,Yes
3,1,0,0,0,0.514251,0,0,-0.746535,-0.194766,False,...,False,False,False,True,False,True,False,False,False,No
4,0,0,0,0,-1.236724,1,1,0.197365,-0.940470,True,...,False,False,True,False,False,False,False,True,False,Yes


In [14]:
cleaned_path = "../data/cleaned_telco.csv"
df_cleaned.to_csv(cleaned_path, index=False)
print(f"Saved cleaned dataset to {cleaned_path}")

Saved cleaned dataset to ../data/processed/cleaned_telco.csv
